# LS_IBM_cpp output viewer

Loads the solver's JSON output and plots the staggered-grid fields, one figure per field.
Each plot is also saved to `PROJECT/plots/<var>/<var>_<iTime>.png`.

The C++ solver writes:
- `coordinates.json` : `{imax, jmax, lx, ly, xu, yu, xv, yv, xp, yp}`
- `dataRDE<N>dt.json`: `{time, U, V, P, phi: {phi_0, ...}, psi}`

Every field is a **flat, row-major** array (`index = i*ny + j`), each on its own staggered grid:

| field | shape | grid |
|---|---|---|
| `U` | `(imax, jmax+1)` | `(xu, yu)` |
| `V` | `(imax+1, jmax)` | `(xv, yv)` |
| `P`, `phi`, `psi` | `(imax+1, jmax+1)` | `(xp, yp)` |

On Sherlock (from `sh_dev`, not the login node): `ml python py-numpy py-matplotlib jupyter`, then launch.

In [ ]:
import json
import os
import re
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# ---- edit these paths ----
PROJECT = '/home/groups/ibattiat/sxia/LS_IBM/LS_IBM_sxia/Example_Project'
COORDS  = os.path.join(PROJECT, 'coordinates.json')            # note: main.cpp writes this to the run CWD
FRAME   = os.path.join(PROJECT, 'output', 'dataRDE1dt.json')   # one of the dataRDE<N>dt.json files

# plots are saved under PROJECT/plots/<var>/<var>_<iTime>.png
PLOTS = os.path.join(PROJECT, 'plots')
# iTime = the integer N in dataRDE<N>dt.json (used in each plot's filename)
_m = re.search(r'dataRDE(\d+)dt', os.path.basename(FRAME))
iTime = int(_m.group(1)) if _m else 0

def load(p):
    with open(p) as f:
        return json.load(f)

coords = load(COORDS)
data   = load(FRAME)
imax, jmax = coords['imax'], coords['jmax']
print(f"grid {imax} x {jmax},  t = {data.get('time', 0.0)},  iTime = {iTime}")
print('fields:', ['U','V','P','psi'] + sorted(data['phi']))

In [ ]:
def grid2d(flat, nx, ny):
    """Flat row-major (index=i*ny+j) -> 2D array shaped (nx, ny) == [x, y]."""
    a = np.asarray(flat, dtype=float)
    assert a.size == nx * ny, f'expected {nx*ny} values, got {a.size}'
    return a.reshape(nx, ny)

xu, yu = coords['xu'], coords['yu']   # len imax,   jmax+1
xv, yv = coords['xv'], coords['yv']   # len imax+1, jmax
xp, yp = coords['xp'], coords['yp']   # len imax+1, jmax+1

U   = grid2d(data['U'],   imax,     jmax + 1)
V   = grid2d(data['V'],   imax + 1, jmax)
P   = grid2d(data['P'],   imax + 1, jmax + 1)
psi = grid2d(data['psi'], imax + 1, jmax + 1)
phi = {k: grid2d(v, imax + 1, jmax + 1) for k, v in data['phi'].items()}

In [ ]:
def show(x, y, f_xy, var, title, cmap='viridis', grain=True, vmin=None, vmax=None):
    """One field, one figure. f_xy is [x, y]; pcolormesh wants (len(y), len(x)) -> transpose.
    Saves to PROJECT/plots/<var>/<var>_<iTime>.png (folder created if needed).
    vmin/vmax fix the colorbar range; if None, use the plotted data's min/max
    (works for masked arrays -- masked cells are ignored)."""
    
    min = f_xy.min()
    max = f_xy.max()
    if vmin is None:
        lim = np.max(np.abs(np.array([min, max])))
        vmin = -lim
        vmax = lim
        

    fig, ax = plt.subplots(figsize=(7, 4))
    pc = ax.pcolormesh(x, y, f_xy.T, shading='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    if grain:  # grain surface = psi==0 contour (psi lives on xp/yp)
        ax.contour(xp, yp, psi.T, levels=[0.0], colors='k', linewidths=1.0)
    ax.set_title(f"{title}   (t = {data.get('time', 0.0):.6g})")
    ax.set_aspect('equal')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    fig.colorbar(pc, ax=ax, shrink=0.85)

    folder = os.path.join(PLOTS, var)
    os.makedirs(folder, exist_ok=True)
    out = os.path.join(folder, f"{var}_{iTime}.png")
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print('saved', out)
    plt.show()

In [ ]:
show(xu, yu, U,   'U',   'U',  'RdBu_r',  vmin=None, vmax=None)
show(xv, yv, V,   'V',   'V',  'RdBu_r',  vmin=None, vmax=None)
show(xp, yp, P,   'P',   'P',  'viridis', vmin=None, vmax=None)

In [ ]:
show(xp, yp, psi, 'Psi', 'psi', 'coolwarm', vmin=None, vmax=None)

In [ ]:
psi_binary = np.ma.masked_where(psi <= 0, psi)
psi_binary = np.ma.masked_where(psi_binary <= 0, np.ones(psi.shape))
show(xp, yp, psi_binary, 'Psi_binary', 'psi (binary)', 'coolwarm', vmin=None, vmax=None)

In [ ]:
# one figure per chemical species -> plots/phi_0/phi_0_<iTime>.png, etc.
for name in sorted(phi):
    show(xp, yp, phi[name], name, name, 'magma', vmin=None, vmax=None)

## Sanity check for the frozen (no-LS) milestone

With the current setup (no solve yet, geometry frozen) you should see:
- **`psi`**: concentric rings around the centered grain, clean circle at `psi==0`.
- **`U`**: ~`uinflow` everywhere except a zero disk where the grain sits.
- **`V`, `P`**: all zero.
- **`phi_0`** ~ 1 in the fluid, 0 inside the grain; `phi_1`/`phi_2` ~ 0.

If that matches, the whole `VariableNonDim` setup pipeline is working end to end.